[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/23_cross_attention_interview.ipynb)

# 🟠 Solution: Multi-Head Cross-Attention（面试版）

## 📋 题目描述

**难度：** Medium

实现多头交叉注意力（nn.Module）。

交叉注意力让解码器关注编码器输出：Q 来自一个序列，K/V 来自另一个序列，不使用因果掩码。

**签名:** `MultiHeadCrossAttention(d_model, num_heads)`（nn.Module）

**前向传播:** `forward(x_q, x_kv) -> Tensor`
- `x_q` — 查询输入 (B, S_q, d_model)
- `x_kv` — 键/值输入 (B, S_kv, d_model)

**返回:** 注意力输出 (B, S_q, d_model)

**约束:**
- 使用独立的 W_q、W_k、W_v、W_o 线性投影
- Q 和 KV 可以有不同的序列长度

**提示：** Q 来自 `x_q`，K/V 来自 `x_kv`。投影 → reshape 为 `(B, H, S, d_k)` → 缩放点积（无因果遮蔽）→ 拼接各头 → `W_o`。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        B, S_q, _ = x_q.shape
        S_kv = x_kv.shape[1]

        # ---- Step 1: 投影 ----
        q = self.W_q(x_q)  # (B, S_q, d_model)
        k = self.W_k(x_kv)  # (B, S_kv, d_model)
        v = self.W_v(x_kv)  # (B, S_kv, d_model)

        # ---- Step 2: 拆分多头 ----
        # 关键操作：view + transpose
        # (B, S, d_model) → (B, S, H, d_k) → (B, H, S, d_k)
        q = q.view(B, S_q, self.num_heads, self.d_k).transpose(1, 2)
        k = k.view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        v = v.view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)

        # ---- Step 3: 手写 softmax ----
        # 面试可能要求不使用 torch.softmax
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # 数值稳定 softmax：减去最大值防溢出
        scores_max = scores.max(dim=-1, keepdim=True).values
        exp_scores = torch.exp(scores - scores_max)
        weights = exp_scores / exp_scores.sum(dim=-1, keepdim=True)

        # ---- Step 4: 加权求和 + 合并 ----
        attn = torch.matmul(weights, v)
        out = attn.transpose(1, 2).contiguous().view(B, S_q, -1)
        return self.W_o(out)

# 面试常见追问：
# Q: 交叉注意力 vs 自注意力？
# A: 自注意力：Q=K=V=x（同一个序列）
#    交叉注意力：Q 来自解码器，K/V 来自编码器（不同序列）
# Q: 为什么除以 sqrt(d_k)？
# A: 防止点积值过大。假设 Q,K 各分量独立标准正态，
#    Q·K 的方差 = d_k，标准差 = sqrt(d_k)。
#    除以 sqrt(d_k) 使方差回到 1，softmax 不会饱和。


In [ ]:
mha = MultiHeadCrossAttention(d_model=64, num_heads=8)
x_q = torch.randn(2, 10, 64)   # decoder query: 10 tokens
x_kv = torch.randn(2, 20, 64)  # encoder kv: 20 tokens
print('Output:', mha(x_q, x_kv).shape)


In [ ]:
from torch_judge import check
check('cross_attention')


## 📝 核心思路总结

1. **Q/K/V 分离**：Q 来自解码器，K/V 来自编码器，各自独立投影
2. **多头拆分**：`view + transpose` 将 d_model 拆为 num_heads × d_k
3. **缩放点积**：`QK^T / sqrt(d_k)` + softmax + 加权 V
4. **无因果掩码**：交叉注意力允许 Q 关注所有 KV 位置
